In [1]:
import torch
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

ds = load_dataset("JacobLinCool/VoiceBank-DEMAND-16k")
print(ds)

test = ds['test']

c:\Users\astag\Documents\Master Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on cuda


DatasetDict({
    train: Dataset({
        features: ['id', 'clean', 'noisy'],
        num_rows: 11572
    })
    test: Dataset({
        features: ['id', 'clean', 'noisy'],
        num_rows: 824
    })
})


In [2]:
from modules import DeepFilterNet2, Config

sample_rate = 16000
n_fft = 512
f_df = 5000
C = 64
N = 5
lambdaspec = 1e3
lambdamr = 5e2

win_length = int(sample_rate / 1000 * 20)
window = torch.hann_window(win_length).to(device)
hop_length = win_length // 2
freqs = torch.fft.rfftfreq(
    n_fft,
    d=1 / sample_rate,
)
# Remove DC and Nyquist bins
freqs = freqs[1:-1]
df_indices = freqs <= f_df
N_df = df_indices.sum().item()

config = Config(
    sample_rate=sample_rate,
    n_fft=n_fft,
    f_df=f_df,
    N=N,
    lambdamr=lambdamr,
    lambdaspec=lambdaspec,
    hop_length=hop_length,
    win_length=win_length,
    df_indices=df_indices,
    N_df=N_df
)

model = DeepFilterNet2(C, N, N_df).to(device)

## LOAD MODEL

In [3]:
checkpoint = torch.load(
    "checkpoints/best.pt",
    map_location=device,
)

model.load_state_dict(checkpoint["model_state_dict"])

# IMPORTANT for inference
model.eval()

print(
    f"Loaded model from epoch {checkpoint['epoch']} "
    f"(validation loss: {checkpoint['val_loss']:.4e})"
)

Loaded model from epoch 40 (validation loss: 3.6370e+07)


## Inference on Test dataset

In [4]:
from tqdm import tqdm
from fonctions import process, apply_deep_filter
from pesq import pesq
from pystoi import stoi
from pathlib import Path
import torchaudio

pesq_enhanced = []
pesq_noisy = []

stoi_enhanced = []
stoi_noisy = []

enhanced_dir = Path("enhanced")
enhanced_dir.mkdir(parents=True, exist_ok=True)

for signal in tqdm(test):
    clean = signal['clean'].get_all_samples().data.float().to(device)
    noisy = signal['noisy'].get_all_samples().data.float().to(device)
    Xnorm, Xdf, ft = process(
            noisy,
            window,
            config,
        )
    
    G_erb, C_df = model(Xnorm, Xdf)
    C_real = C_df[:, :config.N]
    C_imag = C_df[:, config.N:]
    C_df_comp = torch.complex(C_real, C_imag)

    Y_df = apply_deep_filter(ft, G_erb, C_df_comp, config)

    y = torch.istft(
            Y_df,
            n_fft=config.n_fft,
            hop_length=config.hop_length,
            win_length=config.win_length,
            window=window,
            length=noisy.shape[-1],
        )

    clean_np = clean.detach().squeeze().cpu().numpy()
    noisy_np = noisy.detach().squeeze().cpu().numpy()
    enhanced_np = y.detach().squeeze().cpu().numpy()

    pesq_enhanced.append(pesq(config.sample_rate, clean_np, enhanced_np, "wb"))
    pesq_noisy.append(pesq(config.sample_rate, clean_np, noisy_np, "wb"))


    stoi_noisy.append(stoi(clean_np, noisy_np, config.sample_rate, extended=False))
    stoi_enhanced.append(stoi(clean_np, enhanced_np, config.sample_rate, extended=False))

    output_path = enhanced_dir / f"{signal['id']}.wav"

    torchaudio.save(
            str(output_path),
            y.detach().cpu(),
            config.sample_rate
        )
    

100%|██████████| 824/824 [04:29<00:00,  3.06it/s]


In [5]:
from pathlib import Path
import sys 
import subprocess

code_dir = Path.cwd()
dnsmos_repo = code_dir / "DNSMOS" / "DNS-Challenge" / "DNSMOS"

result = subprocess.run(
    [
        sys.executable,
        "dnsmos_local.py",
        "-t", str(code_dir / "enhanced"),
        "-o", str(code_dir / "dnsmos.csv"),
    ],
    cwd=dnsmos_repo,
)

In [8]:
import pandas as pd
import numpy as np 

df_enhanced = pd.read_csv("dnsmos.csv")
df_noisy = pd.read_csv("dnsmos_noisy.csv")


metrics = ["OVRL", "SIG", "BAK"]

for metric in metrics:
    enhanced_mean = df_enhanced[metric].mean()
    noisy_mean = df_noisy[metric].mean()
    improvement = enhanced_mean - noisy_mean

    print(f"{metric}:")
    print(f"  Noisy:     {noisy_mean:.4f}")
    print(f"  Enhanced:  {enhanced_mean:.4f}")
    print(f"  Δ:         {improvement:+.4f}")

# PESQ
pesq_enhanced_mean = np.mean(pesq_enhanced)
pesq_noisy_mean = np.mean(pesq_noisy)

print("PESQ:")
print(f"  Noisy:     {pesq_noisy_mean:.4f}")
print(f"  Enhanced:  {pesq_enhanced_mean:.4f}")
print(f"  Δ:         {pesq_enhanced_mean - pesq_noisy_mean:+.4f}")

# STOI
stoi_enhanced_mean = np.mean(stoi_enhanced)
stoi_noisy_mean = np.mean(stoi_noisy)

print("STOI:")
print(f"  Noisy:     {stoi_noisy_mean:.4f}")
print(f"  Enhanced:  {stoi_enhanced_mean:.4f}")
print(f"  Δ:         {stoi_enhanced_mean - stoi_noisy_mean:+.4f}")

OVRL:
  Noisy:     2.6973
  Enhanced:  3.1422
  Δ:         +0.4449
SIG:
  Noisy:     3.3456
  Enhanced:  3.4610
  Δ:         +0.1153
BAK:
  Noisy:     3.1264
  Enhanced:  3.9685
  Δ:         +0.8421
PESQ:
  Noisy:     1.9707
  Enhanced:  2.7820
  Δ:         +0.8114
STOI:
  Noisy:     0.9210
  Enhanced:  0.9385
  Δ:         +0.0175


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load DNSMOS results
df_enhanced = pd.read_csv("dnsmos.csv")
df_noisy = pd.read_csv("dnsmos_noisy.csv")

# Extract utterance ID from filename
df_enhanced["id"] = df_enhanced["filename"].apply(
    lambda x: Path(x).stem
)

df_noisy["id"] = df_noisy["filename"].apply(
    lambda x: Path(x).stem
)

# Keep and rename DNSMOS columns
enhanced_dns = df_enhanced[
    ["id", "OVRL", "SIG", "BAK"]
].rename(columns={
    "OVRL": "dnsmos_ovrl_enhanced",
    "SIG": "dnsmos_sig_enhanced",
    "BAK": "dnsmos_bak_enhanced",
})

noisy_dns = df_noisy[
    ["id", "OVRL", "SIG", "BAK"]
].rename(columns={
    "OVRL": "dnsmos_ovrl_noisy",
    "SIG": "dnsmos_sig_noisy",
    "BAK": "dnsmos_bak_noisy",
})

# PESQ/STOI results generated by your evaluation loop
objective = pd.DataFrame({
    "id": [signal["id"] for signal in test],
    "pesq_noisy": pesq_noisy,
    "pesq_enhanced": pesq_enhanced,
    "stoi_noisy": stoi_noisy,
    "stoi_enhanced": stoi_enhanced,
})

# Merge everything by utterance ID
results = (
    objective
    .merge(noisy_dns, on="id", how="inner")
    .merge(enhanced_dns, on="id", how="inner")
)
results["pesq_improvement"] = (
    results["pesq_enhanced"] - results["pesq_noisy"]
)

results["stoi_improvement"] = (
    results["stoi_enhanced"] - results["stoi_noisy"]
)

results["dnsmos_ovrl_improvement"] = (
    results["dnsmos_ovrl_enhanced"]
    - results["dnsmos_ovrl_noisy"]
)

results["dnsmos_sig_improvement"] = (
    results["dnsmos_sig_enhanced"]
    - results["dnsmos_sig_noisy"]
)

results["dnsmos_bak_improvement"] = (
    results["dnsmos_bak_enhanced"]
    - results["dnsmos_bak_noisy"]
)

results.head()

,id,pesq_noisy,pesq_enhanced,stoi_noisy,stoi_enhanced,dnsmos_ovrl_noisy,dnsmos_sig_noisy,dnsmos_bak_noisy,dnsmos_ovrl_enhanced,dnsmos_sig_enhanced,dnsmos_bak_enhanced
0,p232_001,2.927393,3.234224,0.897550,0.906500,3.227739,3.621405,3.897710,3.292520,3.580823,4.057775
1,p232_002,3.059851,3.722148,0.969538,0.973450,3.265373,3.685854,3.796284,3.483589,3.673317,4.194746
2,p232_003,2.814329,3.444052,0.971748,0.978518,3.106473,3.539224,3.767159,3.368031,3.600442,4.163914
3,p232_005,1.328514,2.027829,0.881930,0.906282,2.481212,3.532305,2.509196,3.467341,3.709413,4.133673
4,p232_006,2.203033,3.003231,0.965141,0.956624,2.926365,3.651940,3.223305,3.340101,3.604335,4.091100


In [11]:
summary = results.drop(columns="id").agg(["mean", "std"]).T

summary.round(4)

,mean,std
pesq_noisy,1.9707,0.7541
pesq_enhanced,2.7820,0.7039
stoi_noisy,0.9210,0.0710
stoi_enhanced,0.9385,0.0566
dnsmos_ovrl_noisy,2.6973,0.5191
dnsmos_sig_noisy,3.3456,0.5221
dnsmos_bak_noisy,3.1264,0.7369
dnsmos_ovrl_enhanced,3.1422,0.2174
dnsmos_sig_enhanced,3.4610,0.1778
dnsmos_bak_enhanced,3.9685,0.1806


## Save results

In [12]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(results_dir / "fp32_results.csv", index=False)
summary.to_csv(results_dir / "fp32_summary.csv")